# 05: Expanded Dataset Pipeline (138 Funds)

Scales the recommendation engine from the original 6 hand-picked funds to
138 funds across 7 genuine categories, sourced from a daily-updated
AMFI-derived dataset (GitHub: InertExpert2911/Mutual_Fund_Data).

This is the cleaned, single-pass version of the expansion work. Tonight's
actual process found two real data-quality bugs *after* running the CAGR
pipeline once (a costly ~10 min rerun each time they were found) -- this
version applies both fixes to the raw NAV data upfront, so the slow CAGR
loop only needs to run once.

Requires `calculate_cagr()` and `calculate_rolling_cagr()` from
`03_eda_feature_engineering.ipynb` -- either run that notebook first or
paste the two function definitions into the cell below.

Produces the checkpoint CSVs consumed by `06_recommendations.ipynb`.

## Setup

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

`calculate_cagr()` / `calculate_rolling_cagr()` -- same functions validated
in `03_eda_feature_engineering.ipynb`, unchanged. Included here so this
notebook can run standalone.

In [2]:
def calculate_cagr(df):
    nav_start = df['nav'].iloc[0]
    nav_end = df['nav'].iloc[-1]

    date_start = df['date'].iloc[0]
    date_end = df['date'].iloc[-1]

    years = (date_end - date_start).days / 365.25  # accounting for leap years

    return (nav_end / nav_start) ** (1 / years) - 1


def calculate_rolling_cagr(df, window_years):
    df = df.copy()
    df = df.set_index('date')

    rolling_cagr = []
    dates = []

    for i in range(len(df)):
        start_date = df.index[i]
        # DateOffset shifts in actual calendar years so windows are year-wise, not day-count approximations
        target_end_date = start_date + pd.DateOffset(years=window_years)

        if target_end_date > df.index[-1]:
            break
        # searchsorted does a binary search and returns the closest available date near target
        end_idx = df.index.searchsorted(target_end_date)

        start_nav = df['nav'].iloc[i]
        end_nav = df['nav'].iloc[end_idx]

        actual_years = (df.index[end_idx] - start_date).days / 365.25
        cagr = (end_nav / start_nav) ** (1 / actual_years) - 1
        rolling_cagr.append(cagr)
        dates.append(start_date)
    return pd.Series(rolling_cagr, index=dates)

## Load fund selection + raw NAV data

140 funds selected across 7 target categories (large-cap, mid-cap, small-cap, corporate bond, short-duration debt, hybrid, index) via `Scheme_Category` substring matching on the Direct+Growth-filtered scheme metadata, then sampled ~20 per category for genuine diversity (not cherry-picked).

In [3]:
nav_selected = pd.read_csv('../Data/external/nav_selected_140funds.csv', parse_dates=['date'])
final_selection = pd.read_csv('../Data/external/final_selection_140funds.csv')

nav_selected = nav_selected.sort_values(['Scheme_Code', 'date']).reset_index(drop=True)
name_lookup = final_selection[['Scheme_Code', 'Scheme_NAV_Name']].drop_duplicates()
category_lookup = final_selection[['Scheme_Code', 'Scheme_Category']].drop_duplicates()

print(nav_selected.shape)
print(nav_selected['Scheme_Code'].nunique(), "funds")

(286236, 3)
140 funds


## Data-quality fixes (applied upfront)

Two real issues were found in this dataset while validating results:

1. **Two funds (Scheme_Code 148265, 148313) have `nav == 0.0` for every
   single row** -- no salvageable data, not just a leading placeholder.
   Dropped entirely.
2. **Invesco India Short Duration Fund (Scheme_Code 120560) has a
   corrupted NAV value on 2013-04-22** that inflates its NAV scale by
   roughly 100x from that date onward (likely a decimal/unit error at the
   source). This produced a fabricated ~257% single-year CAGR when first
   discovered. Fixed by dropping the pre-2013-04-22 segment, keeping only
   the internally-consistent later data.

Both fixes are applied directly to `nav_selected` before the CAGR loop
runs, rather than patched in after the fact.

In [4]:
# Fix 1: drop the two funds with entirely zero NAV history
ZERO_NAV_FUNDS = [148265, 148313]
nav_selected = nav_selected[~nav_selected['Scheme_Code'].isin(ZERO_NAV_FUNDS)]

# Fix 2: drop Invesco India Short Duration Fund's corrupted pre-2013-04-22 segment
INVESCO_SHORT_DURATION_CODE = 120560
invesco_cutoff = pd.Timestamp('2013-04-22')
nav_selected = nav_selected[
    ~((nav_selected['Scheme_Code'] == INVESCO_SHORT_DURATION_CODE) & (nav_selected['date'] < invesco_cutoff))
]

# General safety net: drop any remaining non-positive NAV rows (e.g. leading
# placeholder zeros on other funds, same class of issue as the two dropped funds)
nav_selected = nav_selected[nav_selected['nav'] > 0].copy()
nav_selected = nav_selected.sort_values(['Scheme_Code', 'date']).reset_index(drop=True)

print(nav_selected.shape)
print(nav_selected['Scheme_Code'].nunique(), "funds remaining (expect 138)")

(283145, 3)
138 funds remaining (expect 138)


## CAGR pipeline

Computes both the per-fund/window summary (`results_df`: mean/min/max/overall
CAGR) and the row-level rolling CAGR values (`long_df`: one row per valid
start date), in a single pass over the cleaned data. This loop is the slow
part (~10 min for 138 funds, pure-Python row-by-row rolling window
calculation) -- with the data-quality fixes already applied above, it only
needs to run once.

In [5]:
results = []
long_rows = []

for code, group in tqdm(nav_selected.groupby('Scheme_Code'), total=nav_selected['Scheme_Code'].nunique()):
    group = group.sort_values('date').reset_index(drop=True)
    overall_cagr = calculate_cagr(group)

    for window_years in range(1, 11):
        rolling_series = calculate_rolling_cagr(group, window_years)
        if len(rolling_series) == 0:
            continue  # fund's history shorter than this window

        results.append({
            'Scheme_Code': code,
            'window_years': window_years,
            'mean_cagr': rolling_series.mean(),
            'min_cagr': rolling_series.min(),
            'max_cagr': rolling_series.max(),
            'overall_cagr': overall_cagr
        })

        for start_date, cagr in rolling_series.items():
            long_rows.append({
                'Scheme_Code': code,
                'window_years': window_years,
                'start_date': start_date,
                'cagr': cagr
            })

results_df = pd.DataFrame(results)
long_df = pd.DataFrame(long_rows)

print(results_df.shape)
print(long_df.shape)

# Sanity check -- should be clean now that bad data was removed upfront
print(results_df.isin([float('inf'), float('-inf')]).sum().sum(), "inf values")
print(results_df.isna().sum().sum(), "NaN values")

100%|██████████| 138/138 [04:29<00:00,  1.95s/it]


(935, 6)
(1494509, 4)
0 inf values
0 NaN values


## Checkpoint: raw CAGR results

In [6]:
results_df.to_csv('../Data/external/results_df_138funds.csv', index=False)
long_df.to_csv('../Data/external/long_df_138funds.csv', index=False)

## Category encoding

Switched from per-fund one-hot encoding (used in the 6-fund Phase 5 model)
to category-based encoding, since one-hot per fund at 138 funds would be
unusably sparse (~10 rows per fund). `Scheme_Category` has some naming
variants from the source data (e.g. singular vs plural "Scheme(s)") --
merged before encoding so equivalent categories don't get split into
separate dummy columns.

In [7]:
long_df = long_df.merge(category_lookup, on='Scheme_Code', how='left')

# Merge the corporate bond naming variant into one consistent label
long_df['Scheme_Category'] = long_df['Scheme_Category'].replace(
    'Income/Debt Oriented Schemes - Corporate Bond Fund',
    'Debt Scheme - Corporate Bond Fund'
)

print(long_df['Scheme_Category'].nunique(), "categories (expect 7)")
print(long_df['Scheme_Category'].isna().sum(), "NaN categories (expect 0)")

long_df_encoded = pd.get_dummies(long_df, columns=['Scheme_Category'], prefix='category')
print(long_df_encoded.shape)

7 categories (expect 7)
0 NaN categories (expect 0)
(1494509, 11)


## Quantile regression (ML-based risk estimate)

Same approach as the 6-fund Phase 5 model (`GradientBoostingRegressor`,
`loss='quantile'`, `alpha=0.05`, tuned previously), but trained on
`window_years` + one-hot `category` instead of per-fund dummies. Time-based
80/20 split (not random) to avoid leaking near-duplicate overlapping
windows between train and test.

**Known limitation**: since the model has no fund-level feature, every fund
in the same category at the same `window_years` receives an *identical*
predicted risk value. Historical `min_cagr` remains the fund-specific risk
measure; `predicted_min_cagr` is a category-level estimate.

In [8]:
from sklearn.ensemble import GradientBoostingRegressor

category_cols = [c for c in long_df_encoded.columns if c.startswith('category_')]
feature_cols = ['window_years'] + category_cols

x = long_df_encoded[feature_cols]
y = long_df_encoded['cagr']

cutoff_date = long_df_encoded['start_date'].quantile(0.8)
train_mask = long_df_encoded['start_date'] < cutoff_date
x_train, y_train = x[train_mask], y[train_mask]
x_test, y_test = x[~train_mask], y[~train_mask]

print(cutoff_date)
print(x_train.shape, x_test.shape)

quantile_model = GradientBoostingRegressor(loss='quantile', alpha=0.05, n_estimators=100, random_state=42)
quantile_model.fit(x_train, y_train)

2021-02-03 00:00:00
(1195559, 8) (298950, 8)


,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'quantile'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"alpha alpha: float, default=0.9The alpha-quantile of the huber loss function and the quantileloss function. Only if ``loss='huber'`` or ``loss='quantile'``.Values must be in the range `(0.0, 1.0)`.",0.05
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3


Predict for every row, aggregate to one prediction per fund/window, and
merge into `results_df` alongside the historical figures.

Note: raw test-set MAE against individual `cagr` values looks misleadingly
high for a quantile model -- it's deliberately predicting the 5th
percentile, not the mean, so it's *supposed* to sit below most individual
values. The real validation is comparing the aggregated `predicted_min_cagr`
against historical `min_cagr` per fund/window (below), not raw MAE.

In [9]:
long_df_encoded['predicted_min_cagr'] = quantile_model.predict(x)

predicted_summary = long_df_encoded.groupby(['Scheme_Code', 'window_years'])['predicted_min_cagr'].mean().reset_index()

results_df = results_df.merge(predicted_summary, on=['Scheme_Code', 'window_years'], how='left')

print(results_df.shape)
print(results_df['predicted_min_cagr'].isna().sum(), "NaN (expect 0)")

# Spot check: does predicted_min_cagr track historical min_cagr sensibly?
results_df[['Scheme_Code', 'window_years', 'min_cagr', 'predicted_min_cagr']].head(10)

(935, 7)
0 NaN (expect 0)


,Scheme_Code,window_years,min_cagr,predicted_min_cagr
0,118269,1,-0.201841,-0.097038
1,118269,2,-0.052699,-0.010862
2,118269,3,0.005373,0.029497
3,118269,4,0.052588,0.056677
4,118269,5,0.026134,0.063750
5,118269,6,0.081474,0.081018
6,118269,7,0.089591,0.100192
7,118269,8,0.115664,0.104756
8,118269,9,0.130357,0.113364
9,118269,10,0.128239,0.117770


## Checkpoint: results with ML predictions

In [10]:
results_df.to_csv('../Data/external/results_df_138funds_with_predictions.csv', index=False)

## Build `risk_return_df`

Rename historical columns and attach readable fund names -- the shape
`recommend_fund()` / `advise_investment()` expect.

In [11]:
risk_return_df = results_df.rename(columns={
    'mean_cagr': 'mean',
    'min_cagr': 'min'
})
risk_return_df = risk_return_df.merge(name_lookup, on='Scheme_Code', how='left')
risk_return_df = risk_return_df.rename(columns={'Scheme_NAV_Name': 'fund'})

print(risk_return_df.shape)
print(risk_return_df['fund'].isna().sum(), "NaN (expect 0)")

(935, 8)
0 NaN (expect 0)


## Short-history filter

**Why this exists:** funds whose entire NAV history starts after 2018 have
all their rolling-window CAGR figures computed *only* from the 2019-2024
period -- an unusually strong stretch for Indian equities (COVID recovery
+ broader bull run). This inflates `mean_cagr`/`min_cagr` relative to what
a fund with exposure to a full market cycle would show. In this dataset,
46% of funds (63/138) fall entirely after 2018.

**The fix:** require at least some NAV history before 2018 before trusting
a fund's multi-year figures. This is a data-availability gate, not a
statistical patch -- it removes funds that structurally cannot have a
trustworthy multi-year track record.

In [12]:
history_span = nav_selected.groupby('Scheme_Code')['date'].agg(['min', 'max']).reset_index()
history_span['starts_before_2018'] = history_span['min'] < pd.Timestamp('2018-01-01')

risk_return_df = risk_return_df.merge(
    history_span[['Scheme_Code', 'starts_before_2018']], on='Scheme_Code', how='left'
)

risk_return_df_filtered = risk_return_df[risk_return_df['starts_before_2018'] == True].copy()

print(f"Before filter: {risk_return_df['Scheme_Code'].nunique()} funds")
print(f"After filter: {risk_return_df_filtered['Scheme_Code'].nunique()} funds")
print(risk_return_df_filtered['min'].isna().sum(), "NaN in 'min' (expect 0)")

Before filter: 126 funds
After filter: 75 funds
0 NaN in 'min' (expect 0)


## Final checkpoint

This is the dataset `06_recommendations.ipynb` and the Flask backend actually load -- historical + ML risk figures, readable fund names, short-history funds already excluded.

In [13]:
risk_return_df_filtered.to_csv('../Data/external/risk_return_df_138funds_filtered_min7yr.csv', index=False)

## Quick validation call

In [14]:
import sys
sys.path.append('../src')
from calculators import advise_investment

advise_investment(risk_return_df_filtered, principal=100000, years=5, penalty=1.0, risk_column='min', n=3)

Top 3 fund(s) recommened for 5 years at penalty weight 1.0 (risk measure: min):

  #1: Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option
      Expected Cagr=0.2455 
      Worst Case Cagr return (min)=0.0234
      Score= 0.2455

  #2: Edelweiss Mid Cap Fund - Direct Plan - Growth Option
      Expected Cagr=0.2381 
      Worst Case Cagr return (min)=0.1192
      Score= 0.2381

  #3: Axis Small Cap Fund - Direct Plan - Growth
      Expected Cagr=0.2158 
      Worst Case Cagr return (min)=0.0497
      Score= 0.2158


Projected value of Rs100000 over 5 years, per recommended fund:
  Nippon India Small Cap Fund - Direct Plan Growth Plan - Bonus Option:
      Balanced estimate:      approx Rs206013.33
      Best case (mean):       approx Rs299745.50
      Worst case (min):   approx Rs112281.16
  Edelweiss Mid Cap Fund - Direct Plan - Growth Option:
      Balanced estimate:      approx Rs233246.36
      Best case (mean):       approx Rs290909.19
      Worst case (min):   app

,fund,mean,min,score
0,Nippon India Small Cap Fund - Direct Plan Grow...,0.245520,0.023438,0.245520
1,Edelweiss Mid Cap Fund - Direct Plan - Growth ...,0.238088,0.119172,0.238088
2,Axis Small Cap Fund - Direct Plan - Growth,0.215831,0.049742,0.215831


## Notes

- **`predicted_min_cagr` granularity**: category-level, not fund-level (see
  quantile regression section above). Possible v2: add each fund's own
  historical `mean_cagr` as a single numeric feature to differentiate within
  a category without reintroducing per-fund sparsity.
- **Penalty slider has diminishing effect at longer horizons**: worst-case
  (`min`) CAGR genuinely turns positive for most funds by mid-length
  windows in this filtered dataset (consistent with the core Phase 3
  finding that downside risk shrinks with holding period) -- so the risk
  penalty has less to act on at `window_years=10` than at `window_years=1`.
  Not a bug, but worth knowing when interpreting results at long horizons.
- **Two funds dropped, one fund corrected** -- see the data-quality fixes
  section above.